## Setup do Ambiente
- Importação de bibliotecas essenciais do PySpark
- Definição dos paths utilizados para as tabelas
- Utilização do catálogo `catalogo`, schemas `silver_db_name`, `gold_db_name`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    explode,
    sequence,
    col,
    year,
    quarter,
    month,
    weekofyear,
    dayofmonth,
    dayofweek,
    when
)
from pyspark.sql.types import DateType


### Definição de Variáveis Globais
- Centralizamos os nomes de catálogos, bancos de dados e caminhos.
- **Boas Práticas:** Evitar "hardcoding" (escrever o caminho diretamente no código várias vezes). Se o nome do catálogo mudar no futuro, alteramos apenas aqui.

In [0]:
catalogo = "medalhao_credit"
silver_db_name = "silver_credit"
gold_db_name = "gold_credit"

In [0]:
spark.sql(f"USE CATALOG {catalogo};")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_db_name};")
spark.sql(f"USE SCHEMA {gold_db_name};")

## Criação da tabela `dm_tempo`

Colunas da tabela `dm_tempo`:

- `sk_tempo`
- `ano`
- `trimestre`
- `mes`
- `semana_do_ano`
- `dia`
- `dia_da_semana_num`
- `dia_da_semana_nome`
- `mes_nome`
- `eh_fim_de_semana`

In [0]:
data_inicio = '2016-01-01'
data_fim = '2019-01-01'

df_datas = (
    spark.createDataFrame([(data_inicio, data_fim)], ['data_inicio', 'data_fim'])
    .select(explode(sequence(col('data_inicio').cast(DateType()), col('data_fim').cast(DateType()))).alias('sk_tempo'))
    .withColumn('ano', year(col('sk_tempo')))
    .withColumn('trimestre', quarter(col('sk_tempo')))
    .withColumn('mes', month(col('sk_tempo')))
    .withColumn('semana_do_ano', weekofyear(col('sk_tempo')))
    .withColumn('dia', dayofmonth(col('sk_tempo')))
    .withColumn('dia_da_semana_num', dayofweek(col('sk_tempo')))
    .withColumn('dia_da_semana_nome', 
        when(col('dia_da_semana_num') == 1, 'Domingo')
        .when(col('dia_da_semana_num') == 2, 'Segunda-feira')
        .when(col('dia_da_semana_num') == 3, 'Terça-feira')
        .when(col('dia_da_semana_num') == 4, 'Quarta-feira')
        .when(col('dia_da_semana_num') == 5, 'Quinta-feira')
        .when(col('dia_da_semana_num') == 6, 'Sexta-feira')
        .when(col('dia_da_semana_num') == 7, 'Sabado')
    )
    .withColumn('mes_nome',
        when(col('mes') == 1, 'Janeiro')
        .when(col('mes') == 2, 'Fevereiro')
        .when(col('mes') == 3, 'Março')
        .when(col('mes') == 4, 'Abril')
        .when(col('mes') == 5, 'Maio')
        .when(col('mes') == 6, 'Junho')
        .when(col('mes') == 7, 'Julho')
        .when(col('mes') == 8, 'Agosto')
        .when(col('mes') == 9, 'Setembro')
        .when(col('mes') == 10, 'Outubro')
        .when(col('mes') == 11, 'Novembro')
        .when(col('mes') == 12, 'Dezembro')
    )
    .withColumn('eh_fim_de_semana', when(col('dia_da_semana_num').isin([1,7]), 'Sim').otherwise('Não'))
)

df_datas.limit(20).display()

df_datas.write.mode('overwrite').saveAsTable(f'{catalogo}.{gold_db_name}.dm_tempo')

## Criação da tabela `ft_chamados`

Colunas da tabela `ft_chamados`:

- `id_chamado`
- `id_cliente`
- `id_atendente`
- `motivo`
- `canal`
- `resolvido`
- `nota_atendimento`
- `categoria_nota`
- `status_canal`
- `valor_custo`

In [0]:
df_chamados_geral = spark.table(f'{catalogo}.{silver_db_name}.ft_chamados_geral')
display(df_chamados_geral.limit(20))

In [0]:
print(f"Colunas de {catalogo}.{silver_db_name}.ft_chamados_geral:\n")
print(f"{df_chamados_geral.columns}\n")
print(f"Schema de {catalogo}.{silver_db_name}.ft_chamados_geral:\n")
df_chamados_geral.printSchema()

In [0]:
df_gold_ft_chamados = df_chamados_geral.select(
    col('id_chamado'), col('id_cliente'),     
    when(col('id_atendente') == -1, None).otherwise(col('id_atendente')).alias('id_atendente'), 
    col('motivo'), col('canal'), col('status_canal'), col('resolvido'), col('nota_atendimento'), col('categoria_nota'), col('valor_custo')
)
df_gold_ft_chamados.limit(15).display()
df_gold_ft_chamados.printSchema()

In [0]:
df_gold_ft_chamados.write.mode('overwrite').saveAsTable(f'{catalogo}.{gold_db_name}.ft_chamados')

# Criação da tabela ft_clientes

In [0]:
df_chamados_silver = spark.table(f"{catalogo}.{silver_db_name}.ft_chamados_geral")

In [0]:
df_clientes_silver = spark.table(f"{catalogo}.{silver_db_name}.ft_clientes")

In [0]:
# "Fatos" do cliente
df_metrics_comportamento = (
    df_chamados_silver
    .groupBy("id_cliente")
    .agg(
        # 1. Volumetria
        F.count("id_chamado").alias("total_chamados_historico"),
        
        # 2. Financeiro (Custo que o cliente gerou para a operação)
        F.round(F.sum("valor_custo"), 2).alias("custo_total_atendimento"),
        
        # 3. Satisfação (Média das notas dele)
        F.round(F.avg("nota_atendimento"), 1).alias("nota_media_satisfacao"),
        
        # 4. Tempo (Quanto tempo ele já perdeu com a gente)
        F.sum("tempo_espera_segundos").alias("tempo_total_espera_seg"),
        
        # 5. Recência (Quando foi a última vez que ele ligou?)
        F.max("hora_abertura_chamado").alias("data_ultimo_contato"),
        
        # 6. Resolutividade (Quantos % dos problemas dele foram resolvidos)
        F.round(
            (F.sum(F.when(F.col("resolvido") == "Sim", 1).otherwise(0)) / F.count("id_chamado")) * 100, 
            2
        ).alias("taxa_resolucao_pessoal")
    )
)

display(df_metrics_comportamento.limit(5))

In [0]:
# Tratamento da Dimensão Cliente (Dados Cadastrais)
df_perfil_cliente = (
    df_clientes_silver
    .select("id_cliente", "nome_cliente", "email_cliente", "regiao", "idade")
    
    # Criando Faixa Etária (Facilita análise de Marketing)
    .withColumn("faixa_etaria", 
                F.when(F.col("idade") < 25, "Jovem (ate 24)")
                .when(F.col("idade").between(25, 40), "Adulto Jovem (25-40)")
                .when(F.col("idade").between(41, 60), "Adulto (41-60)")
                .otherwise("Senior (+60)")
    )
)

display(df_perfil_cliente.limit(5))

In [0]:
df_gold_cliente_360 = (
    df_perfil_cliente.alias("cli")
    # Left Join: todos os clientes, mesmo os que nunca ligaram
    .join(df_metrics_comportamento.alias("fat"), "id_cliente", "left")
    
    # Tratamento de Nulos 
    .fillna(0, subset=["total_chamados_historico", "custo_total_atendimento", "tempo_total_espera_seg"])
    
    .withColumn("perfil_cliente",
                F.when((F.col("total_chamados_historico") > 5) & (F.col("nota_media_satisfacao") < 5), "DETRATOR CRITICO")
                .when((F.col("nota_media_satisfacao") >= 9), "PROMOTOR")
                .when(F.col("total_chamados_historico") == 0, "SILENCIOSO (SEM CONTATO)")
                .otherwise("NEUTRO / ATIVO")
    )
    
    .select(
        "id_cliente",
        "nome_cliente",
        "email_cliente",
        "regiao",
        "idade",
        "faixa_etaria",
        "total_chamados_historico",
        "custo_total_atendimento",
        "nota_media_satisfacao",
        "taxa_resolucao_pessoal",
        "data_ultimo_contato",
        "perfil_cliente",
        F.current_timestamp().alias("data_processamento_gold")
    )
)

display(df_gold_cliente_360)

In [0]:
# Nome da Tabela: fato_cliente_perfil (pois mistura fatos e perfil)
tabela_destino = f"{catalogo}.{gold_db_name}.ft_cliente_perfil"

(
    df_gold_cliente_360.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_destino)
)

print(f"Tabela Gold (Customer 360) criada com sucesso em: {tabela_destino}")